### This notebook uses model distillation

In [11]:
from pathlib import Path

import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, resnet50
from torchvision.models import ResNet18_Weights, ResNet50_Weights
import torch.nn.functional as F

from sklearn.metrics import accuracy_score

In [2]:
# Global config

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0   
DATA_ROOT = Path("rust_dataset") 

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR   = DATA_ROOT / "test"

In [3]:
# Device setup

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps") # for mac
    print("Using Apple MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("Using CUDA")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

print("Device:", DEVICE)

Using Apple MPS
Device: mps


In [4]:
# Reproducibility

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Seed set.")

Seed set.


In [5]:
# Supervised train and val transform --> for resNet

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_tf)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=eval_tf)

In [7]:
train_loader = DataLoader(train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2)

val_loader = DataLoader(train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2)

In [8]:
num_classes = 2

teacher = resnet50(weights=ResNet50_Weights.DEFAULT)
teacher.fc = nn.Linear(teacher.fc.in_features, num_classes)

student = resnet18(weights=ResNet18_Weights.DEFAULT)
student.fc = nn.Linear(student.fc.in_features, num_classes)

teacher = teacher.to(DEVICE)
student = student.to(DEVICE)

# Freeze teacher
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/kuoweitseng/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:37<00:00, 2.74MB/s]


In [12]:
# 3. Distillation loss
class DistillationLoss(nn.Module):
    def __init__(self, temperature=4.0, alpha=0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce = nn.CrossEntropyLoss()

    def forward(self, student_logits, teacher_logits, labels):
        T = self.temperature

        # hard-label loss
        ce_loss = self.ce(student_logits, labels)

        # soft-label loss
        kd_loss = F.kl_div(
            F.log_softmax(student_logits / T, dim=1),
            F.softmax(teacher_logits / T, dim=1),
            reduction="batchmean"
        ) * (T * T)

        # blend
        loss = self.alpha * kd_loss + (1.0 - self.alpha) * ce_loss
        return loss, ce_loss.detach(), kd_loss.detach()

criterion = DistillationLoss(temperature=4.0, alpha=0.7)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4, weight_decay=1e-4)


In [13]:
# 4. Train one epoch
def train_one_epoch(student, teacher, loader, optimizer, criterion, device):
    student.train()

    total_loss = 0.0
    total_ce = 0.0
    total_kd = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            teacher_logits = teacher(images)

        student_logits = student(images)
        loss, ce_loss, kd_loss = criterion(student_logits, teacher_logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_ce += ce_loss.item() * images.size(0)
        total_kd += kd_loss.item() * images.size(0)

        preds = student_logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return {
        "loss": total_loss / total,
        "ce_loss": total_ce / total,
        "kd_loss": total_kd / total,
        "acc": correct / total
    }

# 5. Validation
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0
    ce = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = ce(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return {
        "loss": total_loss / total,
        "acc": correct / total
    }

# 6. Training loop
num_epochs = 15
best_val_acc = 0.0

for epoch in range(num_epochs):
    train_metrics = train_one_epoch(student, teacher, train_loader, optimizer, criterion, DEVICE)
    val_metrics = evaluate(student, val_loader, DEVICE)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"train_acc={train_metrics['acc']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['acc']:.4f}"
    )

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        torch.save(student.state_dict(), "student_resnet18_distilled.pth")

print(f"Best val acc: {best_val_acc:.4f}")

Epoch 01/15 | train_loss=0.2134 | train_acc=0.7793 | val_loss=0.4306 | val_acc=0.9238
Epoch 02/15 | train_loss=0.1816 | train_acc=0.9042 | val_loss=0.4513 | val_acc=0.9537
Epoch 03/15 | train_loss=0.1743 | train_acc=0.9434 | val_loss=0.4508 | val_acc=0.9631
Epoch 04/15 | train_loss=0.1707 | train_acc=0.9749 | val_loss=0.4371 | val_acc=0.9866
Epoch 05/15 | train_loss=0.1699 | train_acc=0.9678 | val_loss=0.4345 | val_acc=0.9827
Epoch 06/15 | train_loss=0.1698 | train_acc=0.9717 | val_loss=0.4389 | val_acc=0.9914
Epoch 07/15 | train_loss=0.1694 | train_acc=0.9749 | val_loss=0.4424 | val_acc=0.9906
Epoch 08/15 | train_loss=0.1704 | train_acc=0.9749 | val_loss=0.4319 | val_acc=0.9725
Epoch 09/15 | train_loss=0.1684 | train_acc=0.9804 | val_loss=0.4320 | val_acc=0.9859
Epoch 10/15 | train_loss=0.1676 | train_acc=0.9827 | val_loss=0.4319 | val_acc=0.9937
Epoch 11/15 | train_loss=0.1668 | train_acc=0.9882 | val_loss=0.4373 | val_acc=0.9937
Epoch 12/15 | train_loss=0.1667 | train_acc=0.9866 | v

### Test with baseline ResNet18 model

In [14]:
print(DEVICE)

mps


In [16]:
baseline_model = resnet18(weights=ResNet18_Weights.DEFAULT)
baseline_model.fc = nn.Linear(baseline_model.fc.in_features, num_classes)
baseline_model = baseline_model.to(DEVICE)


# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-4, weight_decay=1e-4)

# Train one epoch
def train_one_epoch_baseline(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return {
        "loss": total_loss / total,
        "acc": correct / total
    }

# Evaluate
@torch.no_grad()
def evaluate_baseline(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return {
        "loss": total_loss / total,
        "acc": correct / total
    }

In [17]:
# Training loop
num_epochs = 15
best_val_acc = 0.0

for epoch in range(num_epochs):
    train_metrics = train_one_epoch_baseline(
        baseline_model, train_loader, optimizer, criterion, DEVICE
    )
    val_metrics = evaluate_baseline(
        baseline_model, val_loader, criterion, DEVICE
    )

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"train_acc={train_metrics['acc']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['acc']:.4f}"
    )

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        torch.save(baseline_model.state_dict(), "baseline_resnet18.pth")

print(f"Best val acc: {best_val_acc:.4f}")

Epoch 01/15 | train_loss=0.2870 | train_acc=0.8704 | val_loss=0.0562 | val_acc=0.9859
Epoch 02/15 | train_loss=0.0632 | train_acc=0.9851 | val_loss=0.0224 | val_acc=0.9961
Epoch 03/15 | train_loss=0.0331 | train_acc=0.9921 | val_loss=0.0148 | val_acc=0.9976
Epoch 04/15 | train_loss=0.0285 | train_acc=0.9890 | val_loss=0.0115 | val_acc=0.9969


KeyboardInterrupt: 